## ___Updating the photosynthetic pathways___
--------------------

In [1]:
!python --version

Python 3.13.8


The system cannot find the path specified.


In [2]:
import numpy as np
import pandas as pd

In [12]:
# https://www.tern.org.au/news/news-photosynthetic-pathways/
# https://portal.tern.org.au/metadata/TERN/1e16257d-57ae-48dd-bbad-5701e72a9f6d
# TERN is an Australia specific dataset
tern = pd.read_csv(r"../../data/chapter2/TERN/Photosynthetic_Pathways_of_Plants_TERN_v2_19092024.csv", encoding="latin1", 
        usecols=["genus", "speciesEpithet", "family", "photosyntheticPathway_confirmed", "photosyntheticPathway_inferred", "photosyntheticPathway_combined"], na_values='U')#, index_col=["genus", "speciesEpithet"])
tern_meta = pd.read_excel(r"../../data/chapter2/TERN/metadata_Photosynthetic_Pathways_of_Plants_TERN_v2_2024.xlsx", sheet_name="Data_Descriptor")

# extra spaces
tern.loc[:, "genus"] = tern.genus.str.strip()
tern.loc[:, "speciesEpithet"] = tern.speciesEpithet.str.strip()
tern.insert(loc=0, column="binominal", value=tern.genus.str.strip() + ' ' + tern.speciesEpithet.str.strip())

# in TRY, photosynthesis pathway is trait id 22
try_photo = pd.read_csv(r"../../data/chapter2/TRY/photosynthetic_pathways.txt", delimiter='\t', encoding="latin1", low_memory=False, decimal='.', usecols=["Dataset", "SpeciesName",
                        "AccSpeciesName", "OrigValueStr", "TraitID"]).dropna(subset=["AccSpeciesName", "OrigValueStr", "TraitID"]) # records where TraitId is NaN are mostly messy metadata rows

# extra spaces 
try_photo.loc[:, "SpeciesName"] = try_photo.SpeciesName.str.strip()
try_photo.loc[:, "OrigValueStr"] = try_photo.OrigValueStr.str.upper().str.replace('.', '')
# messy photosynthetic pathway information
PHOTOSYNTHETIC_PATHWAY_TYPES = { # try to make this as less messy as possible!!
    "C3": "C3",
    "C3?": "C3?",
    "C4": "C4",
    "C4?": "C4?",
    "CAM": "CAM",
    "CAM?": "CAM?",
    "C3/C4": "C3/C4",
    "C3C4": "C3/C4",
    "C3/CAM": "C3/CAM",
    "C3-CAM": "C3/CAM",
    "C4/CAM": "C4/CAM",
    "C4-CAM": "C4/CAM",
    "C3/C4/CAM": "C3/C4/CAM",
    "3": "C3"
}

try_photo.loc[:, "OrigValueStr"] = try_photo.OrigValueStr.replace(PHOTOSYNTHETIC_PATHWAY_TYPES) # clean up the irregularities in the column
subset_categorical = pd.read_csv(r"../../data/chapter2/FREDv3subset/FRED_subset_categorical.csv")

fred_photo = pd.read_csv(r"../../data/chapter2/FRED/FRED3_Entire_Database_2021.csv", low_memory=False, header=0, skiprows=range(1, 10), encoding="latin1",
                         usecols=("F01286", "F01287", "F00043", "F00004")).dropna(subset=("F01286", "F01287", "F00043")).drop_duplicates()
fred_photo.insert(loc=0, column="binominal", value=fred_photo.F01286.str.strip().str.capitalize() + ' ' + fred_photo.F01287.str.strip().str.lower())

In [5]:
species_of_interest = subset_categorical.binominal.drop_duplicates()
species_of_interest

0         Populus trichocarpa
1             Populus tremula
2            Altingia obovata
3       Cryptocarya chinensis
4      Elaeocarpus sylvestris
                ...          
207              Quercus alba
208             Quercus rubra
210       Pleioblastus amarus
234            Nyssa aquatica
235       Platanus acerifolia
Name: binominal, Length: 203, dtype: object

In [10]:
tern.query("binominal.isin(@species_of_interest)") # that's disappointing

,binominal,genus,speciesEpithet,family,photosyntheticPathway_confirmed,photosyntheticPathway_inferred,photosyntheticPathway_combined
250,Acacia auriculiformis,Acacia,auriculiformis,Fabaceae,C3,NaN,C3
2179,Acacia crassicarpa,Acacia,crassicarpa,Fabaceae,NaN,C3,C3


In [15]:
fred_photo.query("binominal.isin(@species_of_interest)").drop_duplicates() # that's nice

,binominal,F00004,F01286,F01287,F00043
0,Dicranopteris linearis,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",Dicranopteris,linearis,C3
3,Cunninghamia lanceolata,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",Cunninghamia,lanceolata,C3
6,Magnolia baillonii,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",Magnolia,baillonii,C3
11,Acacia auriculiformis,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",Acacia,auriculiformis,C3
15,Gordonia axillaris,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",Gordonia,axillaris,C3
...,...,...,...,...,...
54946,Fraxinus mandshurica,"Wang Y, Gao G, Wang N, Wang Z, Gu J. 2019. Eff...",Fraxinus,mandshurica,C3
54952,Juglans mandshurica,"Wang Y, Gao G, Wang N, Wang Z, Gu J. 2019. Eff...",Juglans,mandshurica,C3
54964,Larix gmelinii,"Wang Y, Gao G, Wang N, Wang Z, Gu J. 2019. Eff...",Larix,gmelinii,C3
55060,Fraxinus mandshurica,"Liu YK, Fan C, Li XW, Ling YH, Zhou YG, Feng M...",Fraxinus,mandshurica,C3


In [10]:
groot = pd.read_csv(r"../../data/chapter2/GRooTFullVersion.csv", encoding="latin1", low_memory=False)
groot.insert(loc=0, column="binominal", value=groot.genus.str.capitalize().str.strip() + ' ' + groot.species.str.lower().str.strip())

In [24]:
groot.genus.str.capitalize().str.strip() + ' ' + groot.species.str.lower().str.strip()

0                  NaN
1                  NaN
2                  NaN
3                  NaN
4                  NaN
              ...     
114217    Inula conyza
114218    Inula conyza
114219    Inula conyza
114220    Inula conyza
114221    Inula conyza
Length: 114222, dtype: object

In [31]:
groot_photo = groot.query("not photosyntheticPathway.isna() and not binominal.isna()").loc[:, ["binominal", "photosyntheticPathway"]].drop_duplicates().reset_index(drop=True)
groot_photo

,binominal,photosyntheticPathway
0,Agropyron cristatum,C3
1,Artemisia tridentata,C3
2,Elymus elymoides,C3
3,Acer saccharum,C3
4,Dacrydium cupressinum,C3
...,...,...
4526,Senecio umbrosus,C3
4527,Tephroseris tenuifolia,C3
4528,Veronica aphylla,C3
4529,Vicia onobrychioides,C3
